# 01. 데이터 확인 · 전처리 · 결합

**음악 아티스트 추천 시스템 팀 프로젝트**

이 노트북은 Last.fm 1K(사용자 청취 기록)와 Spotify Tracks(곡의 음악적 특성)를 정리해서
이후 모든 모델이 공통으로 쓸 파일 두 개를 만든다.

**필요한 데이터만 쓰는 방식**
- Last.fm은 `user_id`, `artist_name` 두 컬럼만 읽는다.
- 두 데이터에 **공통으로 있는 아티스트만** 남겨 모든 모델이 같은 아티스트 집합을 쓰게 한다.
- 청취자가 너무 적은 아티스트와 너무 적게 들은 사용자는 제외한다.

**진행 순서 (체크리스트 내용과 같이)**
1. Last.fm 1K 읽기 및 행 수 확인
2. Spotify 아티스트 분리 및 중복 곡 제거
3. 사용자 × 아티스트 재생 횟수 집계
4. 아티스트 매칭률과 청취 기록 커버리지 기록
5. 청취 수가 적은 사용자·아티스트 제거
6. 재생 횟수 log 변환 → 선호도 점수
7. 아티스트별 오디오 특성 평균 벡터
8. 결과 저장 및 요약

## 0. 설정

데이터 파일 위치와 필터링 기준을 여기서 한 번에 정한다. 

In [ ]:
import csv
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# ---- 필터링 기준 (팀에서 합의한 값으로 수정) ----
MIN_USERS_PER_ARTIST = 5     # 이 수 이상의 사용자가 들은 아티스트만 사용
MIN_ARTISTS_PER_USER = 10    # 이 수 이상의 아티스트를 들은 사용자만 사용
CHUNK_SIZE = 500_000         # Last.fm 파일을 나눠 읽는 단위 

# ---- 저장 위치 ----
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---- 데이터 파일 찾기 ----
SEARCH_DIRS = [Path("/kaggle/input"), Path("."), Path("data")]

def find_file(patterns):
    """여러 폴더에서 패턴에 맞는 첫 번째 파일을 찾는다."""
    for base in SEARCH_DIRS:
        if not base.exists():
            continue
        for pattern in patterns:
            found = sorted(base.rglob(pattern))
            if found:
                return found[0]
    return None

LASTFM_FILE = find_file(["userid-timestamp-artid-artname-traid-traname.tsv", "userid-timestamp*.tsv"])
SPOTIFY_FILE = find_file(["dataset.csv", "spotify_tracks*.csv"])

print("Last.fm 파일 :", LASTFM_FILE)
print("Spotify 파일 :", SPOTIFY_FILE)
assert LASTFM_FILE is not None, "Last.fm 1K 청취 기록 파일(userid-timestamp-...tsv)을 찾을 수 없습니다."
assert SPOTIFY_FILE is not None, "Spotify Tracks 파일(dataset.csv)을 찾을 수 없습니다."

Last.fm 파일 : userid-timestamp-artid-artname-traid-traname.tsv
Spotify 파일 : spotify_tracks.csv.csv


### 아티스트 이름 정규화 함수

두 데이터의 아티스트 이름 표기를 맞추기 위한 함수다. 매칭 품질이 이 함수에 달려 있다.

- `unicodedata.normalize("NFKC", ...)`: 전각/반각 문자 등 유니코드 표기 차이를 통일
- `str.casefold()`: 대소문자 통일 (`lower()`보다 다국어에 강함)
- 연속 공백을 하나로 정리

In [2]:
def clean_name(name):
    """아티스트 이름을 비교 가능한 형태로 정규화한다."""
    if pd.isna(name):
        return ""
    name = unicodedata.normalize("NFKC", str(name))
    name = name.casefold().strip()
    name = re.sub(r"\s+", " ", name)
    return name

# 예시 확인
for s in ["  Radiohead ", "THE BEATLES", "Björk", "Ｍｕｓｅ"]:
    print(repr(s), "→", repr(clean_name(s)))

'  Radiohead ' → 'radiohead'
'THE BEATLES' → 'the beatles'
'Björk' → 'björk'
'Ｍｕｓｅ' → 'muse'


## 1. Last.fm 1K 읽기 및 행 수 확인 · 3. 사용자 × 아티스트 집계

 체크리스트: *Last.fm 1K를 `quoting=csv.QUOTE_NONE`으로 읽고 전체 행 수(약 1,900만 건) 확인*
 체크리스트: *사용자×아티스트 재생 횟수로 집계*

**왜 `QUOTE_NONE`인가?** Last.fm 파일에는 곡 제목에 `"`가 들어간 줄이 있다. pandas 기본 설정은 `"`를
문자열 묶음 기호로 해석해서 여러 줄을 한 줄로 합치기

**한 번만 읽기:** 1,900만 줄을 여러 번 읽지 않도록, 읽으면서 바로 `(user_id, artist_name)`별 청취 횟수로 집계

파일 컬럼 구조: `user_id | timestamp | artist_mbid | artist_name | track_mbid | track_name` → 0번, 3번만 사용

In [3]:
parts = []
total_rows = 0      # 파일 전체 행 수
empty_artist = 0    # 아티스트 이름이 비어 있는 행 수

for chunk in pd.read_csv(
    LASTFM_FILE,
    sep="\t",
    header=None,
    usecols=[0, 3],                  # user_id, artist_name만 읽기
    names=["user_id", "artist_name"],
    dtype=str,
    quoting=csv.QUOTE_NONE,          # 따옴표 해석 끄기 (핵심)
    chunksize=CHUNK_SIZE,            # 메모리 절약을 위해 나눠 읽기
    on_bad_lines="warn",             # 깨진 줄이 있으면 조용히 버리지 않고 경고
):
    total_rows += len(chunk)
    chunk["artist_name"] = chunk["artist_name"].map(clean_name)
    empty_artist += (chunk["artist_name"] == "").sum()
    chunk = chunk[chunk["artist_name"] != ""]
    parts.append(chunk.groupby(["user_id", "artist_name"]).size())

# 조각별 집계를 합쳐 최종 사용자 × 아티스트 재생 횟수 표 만들기
plays_all = (
    pd.concat(parts)
    .groupby(level=[0, 1]).sum()
    .rename("plays")
    .reset_index()
)
del parts

valid_events = int(plays_all["plays"].sum())
print(f"전체 행 수            : {total_rows:,}  (약 1,900만 건이면 정상)")
print(f"아티스트 이름 없는 행 : {empty_artist:,}")
print(f"유효 청취 기록        : {valid_events:,}")
print(f"사용자 수             : {plays_all['user_id'].nunique():,}")
print(f"고유 아티스트 수      : {plays_all['artist_name'].nunique():,}")
print(f"사용자×아티스트 쌍    : {len(plays_all):,}")
plays_all.head()

전체 행 수            : 19,150,868  (약 1,900만 건이면 정상)
아티스트 이름 없는 행 : 0
유효 청취 기록        : 19,150,868
사용자 수             : 992
고유 아티스트 수      : 174,035
사용자×아티스트 쌍    : 898,030


,user_id,artist_name,plays
0,user_000001,2562,26
1,user_000001,310,1
2,user_000001,3582,1
3,user_000001,4 wings,2
4,user_000001,4hero,146


## 2. Spotify 아티스트 분리 및 중복 곡 제거

체크리스트: *Spotify `artists`를 `;` 기준으로 분리하고 `track_id` 중복 제거*

- 이 데이터의 `artists` 컬럼은 `"Ingrid Michaelson;ZAYN"`처럼 **세미콜론으로 구분된 문자열**이다.
  분리하지 않으면 피처링 곡이 `"ingrid michaelson;zayn"`이라는 가상의 아티스트가 되어 매칭에서 빠진다.
- 같은 곡이 장르별로 여러 번 들어 있으므로 `track_id` 기준으로 중복을 제거해야 아티스트 평균이 왜곡되지 않는다.
- `DataFrame.explode()`: 리스트가 들어 있는 칸을 여러 행으로 펼친다.

In [4]:
AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo",
]

spotify = pd.read_csv(SPOTIFY_FILE)
print(f"Spotify 원본 행 수      : {len(spotify):,}")

# track_id 기준 중복 제거
spotify = spotify.drop_duplicates(subset="track_id")
print(f"중복 제거 후 곡 수      : {len(spotify):,}")

# 결측 제거 후 ';' 기준으로 아티스트 분리 → 한 행에 한 아티스트
spotify = spotify.dropna(subset=["artists"])
spotify["artist_name"] = spotify["artists"].str.split(";")
spotify = spotify.explode("artist_name")
spotify["artist_name"] = spotify["artist_name"].map(clean_name)
spotify = spotify[spotify["artist_name"] != ""]

print(f"아티스트 분리 후 행 수  : {len(spotify):,}")
print(f"Spotify 고유 아티스트 수: {spotify['artist_name'].nunique():,}")
spotify[["track_name", "artists", "artist_name"]].head()

Spotify 원본 행 수      : 114,000
중복 제거 후 곡 수      : 89,741
아티스트 분리 후 행 수  : 123,424
Spotify 고유 아티스트 수: 29,811


,track_name,artists,artist_name
0,Comedy,Gen Hoshino,gen hoshino
1,Ghost - Acoustic,Ben Woodward,ben woodward
2,To Begin Again,Ingrid Michaelson;ZAYN,ingrid michaelson
2,To Begin Again,Ingrid Michaelson;ZAYN,zayn
3,Can't Help Falling In Love,Kina Grannis,kina grannis


## 4. 아티스트 매칭률과 청취 기록 커버리지

체크리스트: *아티스트 매칭률과 청취 기록 커버리지 기록*

두 가지 기준으로 본다. **발표 자료(전처리 파트)에 그대로 쓸 숫자다.**
- **아티스트 매칭률**: Last.fm 아티스트 중 Spotify에도 있는 비율
- **청취 기록 커버리지**: 전체 청취 기록 중 매칭된 아티스트의 청취가 차지하는 비율

인기 아티스트는 대부분 매칭되기 때문에 보통 커버리지가 매칭률보다 훨씬 높게 나온다.

In [5]:
lastfm_artists = set(plays_all["artist_name"])
spotify_artists = set(spotify["artist_name"])
common_artists = lastfm_artists & spotify_artists

matched_events = plays_all.loc[plays_all["artist_name"].isin(common_artists), "plays"].sum()
artist_match_rate = len(common_artists) / len(lastfm_artists) * 100
event_coverage = matched_events / valid_events * 100

print(f"Last.fm 고유 아티스트 : {len(lastfm_artists):,}")
print(f"Spotify 고유 아티스트 : {len(spotify_artists):,}")
print(f"공통 아티스트         : {len(common_artists):,}")
print(f"아티스트 매칭률       : {artist_match_rate:.2f}%")
print(f"청취 기록 커버리지    : {event_coverage:.2f}%")

# 사용자별 커버리지: 각 사용자의 청취 중 매칭된 아티스트 비율
plays_all["matched"] = plays_all["artist_name"].isin(common_artists)
plays_all["matched_plays"] = plays_all["plays"].where(plays_all["matched"], 0)
user_sum = plays_all.groupby("user_id")[["plays", "matched_plays"]].sum()
user_cov = user_sum["matched_plays"] / user_sum["plays"] * 100
print("\n[사용자별 커버리지(%) 분포]")
print(user_cov.describe().round(2))

print("\n[청취가 많은 매칭 아티스트 Top 10]")
top_matched = (
    plays_all[plays_all["matched"]]
    .groupby("artist_name")["plays"].sum()
    .sort_values(ascending=False).head(10)
)
print(top_matched.to_string())

Last.fm 고유 아티스트 : 174,035
Spotify 고유 아티스트 : 29,811
공통 아티스트         : 7,662
아티스트 매칭률       : 4.40%
청취 기록 커버리지    : 48.97%

[사용자별 커버리지(%) 분포]
count    992.00
mean      49.44
std       15.53
min        0.00
25%       39.65
50%       49.60
75%       59.92
max       95.72
dtype: float64

[청취가 많은 매칭 아티스트 Top 10]
artist_name
radiohead              115209
the beatles            100338
nine inch nails         84421
coldplay                62251
pink floyd              58561
death cab for cutie     58083
elliott smith           50278
the cure                50207
britney spears          49471
david bowie             49102


## 5. 공통 아티스트만 남기고 청취 수가 적은 사용자·아티스트 제거

체크리스트: *청취 수가 너무 적은 사용자·아티스트 제거*

1. **공통 아티스트만 남긴다** → 협업 필터링과 콘텐츠 기반을 같은 아티스트 집합에서 공정하게 비교하기 위함
2. **청취 사용자가 `MIN_USERS_PER_ARTIST`명 미만인 아티스트 제거** → 데이터가 너무 적어 유사도를 믿을 수 없음
3. **`MIN_ARTISTS_PER_USER`개 미만의 아티스트를 들은 사용자 제거** → train/test로 나눌 기록이 부족함

아티스트를 지우면 사용자 기록이 줄고, 사용자를 지우면 아티스트 청취자가 줄기 때문에
조건이 더 이상 바뀌지 않을 때까지 반복한다.

In [6]:
plays = plays_all.loc[plays_all["matched"], ["user_id", "artist_name", "plays"]].copy()
print(f"공통 아티스트만 남긴 후: 사용자 {plays['user_id'].nunique():,}명, "
      f"아티스트 {plays['artist_name'].nunique():,}명, 쌍 {len(plays):,}개")

for step in range(1, 21):
    before = len(plays)
    artist_users = plays.groupby("artist_name")["user_id"].transform("nunique")
    plays = plays[artist_users >= MIN_USERS_PER_ARTIST]
    user_artists = plays.groupby("user_id")["artist_name"].transform("nunique")
    plays = plays[user_artists >= MIN_ARTISTS_PER_USER]
    if len(plays) == before:
        break

n_users = plays["user_id"].nunique()
n_artists = plays["artist_name"].nunique()
sparsity = 1 - len(plays) / (n_users * n_artists)
kept_events = plays["plays"].sum() / valid_events * 100

print(f"\n필터링 완료 ({step}회 반복)")
print(f"사용자            : {n_users:,}명")
print(f"아티스트          : {n_artists:,}명")
print(f"사용자×아티스트 쌍: {len(plays):,}개")
print(f"남은 청취 기록    : 전체의 {kept_events:.2f}%")
print(f"행렬 희소성       : {sparsity:.4%}")

공통 아티스트만 남긴 후: 사용자 989명, 아티스트 7,662명, 쌍 287,334개

필터링 완료 (2회 반복)
사용자            : 973명
아티스트          : 4,824명
사용자×아티스트 쌍: 281,652개
남은 청취 기록    : 전체의 48.61%
행렬 희소성       : 93.9994%


## 6. 재생 횟수 log 변환 → 선호도 점수

체크리스트: *재생 횟수를 log 변환해 선호도 점수 만들기*

재생 횟수는 한 아티스트를 수천 번 들은 사용자도 있어서 편차가 매우 크다.
그대로 쓰면 소수의 극단값이 유사도 계산을 지배하므로 `log(1 + plays)`로 줄인다.

- `np.log1p(x)`: `log(1 + x)`를 계산 (0일 때도 안전)

In [7]:
plays["preference"] = np.log1p(plays["plays"])

print("[재생 횟수 원본]")
print(plays["plays"].describe().round(2))
print("\n[log 변환 후 선호도 점수]")
print(plays["preference"].describe().round(2))

[재생 횟수 원본]
count    281652.00
mean         33.05
std         175.14
min           1.00
25%           1.00
50%           4.00
75%          16.00
max       26496.00
Name: plays, dtype: float64

[log 변환 후 선호도 점수]
count    281652.00
mean          2.02
std           1.42
min           0.69
25%           0.69
50%           1.61
75%           2.83
max          10.18
Name: preference, dtype: float64


## 7. 아티스트별 오디오 특성 평균 벡터

체크리스트: *아티스트별 오디오 특성 평균 벡터를 만들어 `artist_features.csv`로 저장*

필터링 후 남은 아티스트에 대해서만 Spotify 곡들의 오디오 특성을 평균 낸다.
`n_tracks`(아티스트의 곡 수)도 함께 저장해 두면 곡이 1~2개뿐인 아티스트를 나중에 구분할 수 있다.

> 스케일링(StandardScaler 등)은 콘텐츠 기반 모델 단계에서 한다. 여기서는 원래 값 그대로 저장한다.

In [8]:
final_artists = set(plays["artist_name"])

artist_features = (
    spotify[spotify["artist_name"].isin(final_artists)]
    .groupby("artist_name")
    .agg(n_tracks=("track_id", "nunique"), **{f: (f, "mean") for f in AUDIO_FEATURES})
    .reset_index()
)

print(f"특성 벡터가 만들어진 아티스트: {len(artist_features):,}명 (협업 필터링 아티스트와 동일해야 함)")
assert set(artist_features["artist_name"]) == final_artists
artist_features.head()

특성 벡터가 만들어진 아티스트: 4,824명 (협업 필터링 아티스트와 동일해야 함)


,artist_name,n_tracks,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,"""weird al"" yankovic",15,0.657400,0.57500,-10.201133,0.107140,0.250997,0.000008,0.193360,0.736000,117.368000
1,'til tuesday,1,0.611000,0.64500,-7.394000,0.026400,0.003390,0.000004,0.079300,0.411000,106.639000
2,-m-,8,0.586625,0.60175,-9.585000,0.082187,0.409550,0.014930,0.337025,0.425250,104.326125
3,10 years,4,0.432250,0.81425,-4.368500,0.052850,0.006354,0.000088,0.220875,0.320250,155.636250
4,10-feet,3,0.449667,0.98500,-1.744000,0.102767,0.023434,0.000378,0.279667,0.532333,142.190333


## 8. 결과 저장 및 요약

`outputs/` 폴더에 두 파일을 저장한다. **팀원들은 이후 단계에서 이 파일만 불러오면 된다.**

```python
plays = pd.read_csv("outputs/user_artist_plays.csv")
artist_features = pd.read_csv("outputs/artist_features.csv")
```

In [9]:
plays.to_csv(OUTPUT_DIR / "user_artist_plays.csv", index=False)
artist_features.to_csv(OUTPUT_DIR / "artist_features.csv", index=False)

summary = pd.DataFrame({
    "항목": [
        "Last.fm 전체 행 수", "유효 청취 기록",
        "Last.fm 고유 아티스트", "Spotify 고유 아티스트", "공통 아티스트",
        "아티스트 매칭률(%)", "청취 기록 커버리지(%)",
        "최종 사용자 수", "최종 아티스트 수", "최종 사용자×아티스트 쌍",
        "최종 남은 청취 기록 비율(%)", "행렬 희소성(%)",
    ],
    "값": [
        f"{total_rows:,}", f"{valid_events:,}",
        f"{len(lastfm_artists):,}", f"{len(spotify_artists):,}", f"{len(common_artists):,}",
        f"{artist_match_rate:.2f}", f"{event_coverage:.2f}",
        f"{n_users:,}", f"{n_artists:,}", f"{len(plays):,}",
        f"{kept_events:.2f}", f"{sparsity*100:.2f}",
    ],
})
summary.to_csv(OUTPUT_DIR / "preprocessing_summary.csv", index=False)

print("저장 완료:")
for f in ["user_artist_plays.csv", "artist_features.csv", "preprocessing_summary.csv"]:
    print(" -", OUTPUT_DIR / f)
summary

저장 완료:
 - outputs\user_artist_plays.csv
 - outputs\artist_features.csv
 - outputs\preprocessing_summary.csv


,항목,값
0,Last.fm 전체 행 수,"19,150,868"
1,유효 청취 기록,"19,150,868"
2,Last.fm 고유 아티스트,"174,035"
3,Spotify 고유 아티스트,"29,811"
4,공통 아티스트,"7,662"
5,아티스트 매칭률(%),4.40
6,청취 기록 커버리지(%),48.97
7,최종 사용자 수,973
8,최종 아티스트 수,"4,824"
9,최종 사용자×아티스트 쌍,"281,652"


## 다음 단계로 넘길 내용

- 요약표(`preprocessing_summary.csv`)의 숫자는 **발표 자료의 데이터셋 설명·전처리 슬라이드**에 사용한다.
- 다음 노트북 에서 인기 아티스트, 롱테일 분포, 사용자 정보(성별·나이·국가), 오디오 특성 분포를 시각화한다.
- 필터링 기준(`MIN_USERS_PER_ARTIST`, `MIN_ARTISTS_PER_USER`)을 바꾸면 위 숫자가 모두 바뀐다     

**참고 데이터 출처**
- Last.fm 1K: Ò. Celma, *Music Recommendation and Discovery in the Long Tail*, Springer 2010 — http://ocelma.net/MusicRecommendationDataset/lastfm-1K.html
- Spotify Tracks Dataset (Kaggle, maharshipandya) — https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset